# Week 3 — Monday: Cleaning & Grouping Data

**DATA 202 · Calvin University**

> Hagar, Sarah's Egyptian servant, flees into the desert after mistreatment. There, an angel of the Lord finds her by a spring and speaks to her. She responds:
>
> *"You are a God of seeing,"* she said, *"for truly here I have seen him who looks after me."* — Genesis 16:13
>
> What if collecting, cleaning, and grouping data were themselves a form of *seeing* — of paying attention to people who are otherwise easy to overlook? Theologian Eric Stoddart calls this **"com-veillance"**: not watching *people*, but watching *for*, *with*, and *over* them.

Today's dataset is about people experiencing homelessness — deliberately messy, because real records about real people always are. Cleaning it isn't just a technical chore; it's a decision about how carefully we're willing to look.

**Today's plan (~50 min):**

| Time | Section |
|---|---|
| ~5 min | Load and inspect the messy data |
| ~22 min | Part 1 — Cleaning String Data with regex (SLO 03A) |
| ~18 min | Part 2 — Grouping and Aggregating (SLO 03B) |
| ~5 min | Careful with Aggregations + what's next |

Watch for two kinds of stop-and-check along the way: **🎯 Predict First** (guess before we run the code) and **🙋 Quick Check** (a quick verbal question — no code, no pressure, just think and be ready to answer).

---
## Loading the Data

In [ ]:
import pandas as pd

DATA_PATH = "../../datasets/homeless.csv"
homeless = pd.read_csv(DATA_PATH)
homeless.head()

In [ ]:
homeless.info()

**What to notice:** names carry stray spaces (`"   peter"`), the same city is spelled five different ways (`NEW-YORK`, `new-york`, `New york`, `SF`, `L.A.`...), and `shelter_status` mixes case and phrasing (`SHELTERED`, `shelter , pending`). Nothing here is usable for grouping yet — `groupby("city")` right now would give you a dozen "different" cities that are really the same five.

🙋 **Quick Check:** Before we fix anything — just by skimming the columns above, how many *actual* distinct cities do you think are hiding in this data? Say a number out loud.

---
## Part 1: Cleaning String Data (SLO 03A) · ~22 min

We'll use pandas' **string operations** (`.str.___`) and **regular expressions (regex)** to standardize this.

### Regex: Pattern Matching for Text

A **regular expression** (regex) describes a *pattern* of characters instead of one exact string — a small language for "find text that looks like this."

| Syntax | Meaning | Example | Matches |
|---|---|---|---|
| literal text | the exact characters | `cat` | `cat` |
| `\|` | OR (alternation) | `cat\|dog` | `cat` or `dog` |
| `[A-Z]` | a character class | `[A-Z]+` | one or more uppercase letters |
| `[^...]` | NOT this class | `[^a-zA-Z\s]` | anything that isn't a letter or space |
| `\d` | a digit | `\d{3}` | exactly 3 digits, e.g. `422` |
| `\s` | whitespace | `\s+` | one or more spaces/tabs |
| `.` | any single character | `h.t` | `hat`, `hot`, `h5t`... |
| `*` `+` `?` | quantifiers | `go+gle` | `gogle`, `google`, `gooogle`... |
| `^` ... `$` | start ... end of string | `^shelter$` | the *whole* string is exactly `shelter` |
| `(...)`  | a group, often with `\|` inside | `(shelter\|street)` | `shelter` or `street`, treated as one unit |

You can practice more patterns at [regexone.com](https://regexone.com).

🎯 **Predict First:** Before we run any code — for each pair below, will the pattern match the string? (yes/no)

1. Pattern `shelter.*pending` against `"shelter , pending"`
2. Pattern `^sheltered$` against `"sheltered"`
3. Pattern `^sheltered$` against `"unsheltered"`
4. Pattern `^sheltered$` against `"Sheltered"`

Make your guesses, *then* run the cell below.

In [ ]:
import re

tests = [
    (r"shelter.*pending", "shelter , pending"),
    (r"^sheltered$", "sheltered"),
    (r"^sheltered$", "unsheltered"),
    (r"^sheltered$", "Sheltered"),
]
for pattern, text in tests:
    print(f"{pattern!r:22} vs {text!r:22} -> {bool(re.search(pattern, text))}")

Notice #3 and #4 are both `False`: `^...$` anchors mean the pattern must match the **entire** string (so `unsheltered` fails — it has extra letters before "sheltered"), and regex is **case-sensitive by default** (so `Sheltered` with a capital S doesn't match `sheltered`). This is exactly why we'll lowercase text *before* matching it, in a few cells.

---
### 🔨 Mini-Task A — Write a Regex (~2 min)

Write a regex pattern that matches a string made up **only of letters and spaces** — nothing else (no digits, no punctuation). Test it against the three example strings below using `re.fullmatch()`.

*Hint:* you'll want a character class with `+` (one or more), anchored to the whole string.

In [ ]:
# Your code here
my_pattern = r""  # fill this in

for s in ["New York", "New York3", "New-York"]:
    print(s, "->", bool(re.fullmatch(my_pattern, s)))

---
### Now let's clean for real

In [ ]:
# 1. Strip stray whitespace
homeless["name"] = homeless["name"].str.strip()
homeless["city"] = homeless["city"].str.strip()
homeless[["name", "city"]].head()

🎯 **Predict First:** we're about to (a) turn dashes into spaces, (b) strip out anything that isn't a letter or space, and (c) title-case the result. What will `"L.A."` become? What about `"new-york"`? Take a guess before running the next cell.

In [ ]:
# 2. Standardize separators, drop stray punctuation, then title-case
homeless["name"] = (
    homeless["name"]
    .str.replace(r"[^a-zA-Z\s-]", "", regex=True)   # "chloe." -> "chloe"
    .str.title()                                     # -> "Chloe"
)

homeless["city"] = (
    homeless["city"]
    .str.replace("-", " ", regex=False)               # "new-york" -> "new york"
    .str.replace(r"[^a-zA-Z\s]", "", regex=True)      # "CHICAGO." -> "CHICAGO"
    .str.title()                                       # -> "Chicago"
)

homeless[["name", "city"]].drop_duplicates(subset="city").sort_values("city")

Two cities are still abbreviated — `Sf` and `La` — because title-casing an abbreviation doesn't turn it into a full name. Regex and case rules can only take you so far; sometimes you need an explicit **lookup**.

🙋 **Quick Check:** why *can't* a regex fix `"SF"` → `"San Francisco"`, when regex fixed `"CHICAGO."` → `"Chicago"` just fine? What's fundamentally different about the two problems?

In [ ]:
# 3. Map the remaining abbreviations explicitly
city_map = {"Sf": "San Francisco", "La": "Los Angeles"}
homeless["city"] = homeless["city"].replace(city_map)
homeless["city"].value_counts()

### Two regex, same job

There's rarely only one correct pattern. Both lines below flag the same rows — see if you can tell why before reading the explanation.

In [ ]:
sample = pd.Series(["shelter , pending", "shelter,pending", "shelter  ,  pending", "sheltered"])

version_a = sample.str.contains(r"shelter\s*,\s*pending", regex=True)
version_b = sample.str.contains(r"shelter[ ]*,[ ]*pending", regex=True)

pd.DataFrame({"text": sample, "version_a": version_a, "version_b": version_b})

`\s*` and `[ ]*` do almost the same thing here (zero-or-more spaces) — but `\s` also matches tabs and newlines, while `[ ]` matches only the literal space character. Small choices like this matter once your data gets messier than expected.

In [ ]:
# 4. Standardize shelter_status the same way — case first, then targeted replacements
homeless["shelter_status"] = homeless["shelter_status"].str.strip().str.lower()

homeless["shelter_status"] = (
    homeless["shelter_status"]
    .str.replace(r"^sheltered$", "shelter", regex=True)              # "sheltered" -> "shelter"
    .str.replace(r"shelter\s*,\s*pending", "shelter pending", regex=True)
    .str.replace(r"temporary shelter", "shelter temporary", regex=True)
)
homeless["shelter_status"].value_counts()

---
### 🔨 Mini-Task B — Extend the Pattern (~3 min)

Suppose a few more rows had used the spelling `"temp shelter"` instead of `"temporary shelter"`. Write **one** regex pattern (using `|` for alternation) that matches *either* spelling in a single `.str.replace()` call, and test it below.

In [ ]:
# Your code here
sample = pd.Series(["temporary shelter", "temp shelter", "shelter"])
my_pattern = r""  # fill this in — should match "temporary shelter" OR "temp shelter"

sample.str.replace(my_pattern, "shelter temporary", regex=True)

In [ ]:
# 5. education_level: strip + lowercase collapses most of the mess, then relabel nicely
homeless["education_level"] = (
    homeless["education_level"]
    .str.strip()
    .str.lower()
    .replace({"none": "None", "primary": "Primary", "secondary": "Secondary", "higher": "Higher"})
)
homeless["education_level"].value_counts()

---
### 🔨 Task 1 — Flag a Pattern in Free Text (~5 min)

The `notes` column is unstructured text — but it still holds useful signal. Use `.str.contains()` with a regex to flag every row whose `notes` mention losing a job.

- *Hint:* the notes use different phrasings — `"job loss during pandemic"`, `"lost JOB; looking for work"`. A pattern like `r"job"` with `case=False` will catch both.
- Assign the result (a column of `True`/`False`) to `homeless["job_related"]`.
- **Bonus:** extend your pattern with `|` to *also* flag rows mentioning `"healthcare"`. How many rows match now?

In [ ]:
# Your code here


---
## Part 2: Grouping and Aggregating (SLO 03B) · ~18 min

Sometimes we don't want to look at each row (each person's record) — we want to **summarize groups**:

* How many people are in each city?
* What is the average support amount by shelter status?
* Which group has the highest average years homeless?

This is what `groupby()` and aggregation functions are for — and there's more than one way to ask most of these questions.

In [ ]:
# 1. Grouping by one column: how many people per city?
homeless.groupby("city")["id"].count()

🎯 **Predict First:** before we look at money — which city do you guess has the **most** people in this dataset? Which has the **fewest**? Guess, then check against the output above.

In [ ]:
# 2. Aggregating a numeric column: average monthly support by city
homeless.groupby("city")["monthly_support_usd"].mean()

### Many ways to summarize the same numbers

`.mean()` is only one lens. Passing a **list** of function names to `.agg()` runs several at once, side by side:

In [ ]:
homeless.groupby("shelter_status")["monthly_support_usd"].agg(["mean", "median", "min", "max", "std"])

* **mean** — the arithmetic average; sensitive to a few extreme values.
* **median** — the middle value; barely moves even if one entry is way off.
* **min / max** — the range of what's actually happening in each group.
* **std** — how spread out the values are; a small std means the group is fairly uniform.

🙋 **Quick Check:** if one person's `monthly_support_usd` were mistakenly entered as `50000` instead of `500`, which statistic above would be thrown off the most — the mean or the median? Which would barely notice?

In [ ]:
# 3. Grouping by multiple columns: average years homeless by city AND education level
homeless.groupby(["city", "education_level"])["years_homeless"].mean()

### Naming your aggregations

The dict-style `.agg({...})` you'll see next works well when you're summarizing several *columns*. When you want several *statistics* from the same column with clean, custom output names, **named aggregation** is often nicer:

In [ ]:
# Dict-of-lists style: several statistics on several columns
homeless.groupby("shelter_status").agg({
    "family_size": ["mean", "max"],
    "monthly_support_usd": ["mean", "sum"],
})

In [ ]:
# Named-aggregation style: same idea, flatter and more readable output
homeless.groupby("city").agg(
    avg_support=("monthly_support_usd", "mean"),
    max_support=("monthly_support_usd", "max"),
    n_people=("id", "count"),
)

Both cells above are doing the *same kind* of work — summarizing several statistics per group — just with different syntax. Pick whichever reads more clearly for the summary you're building.

---
### 🔨 Mini-Task C — Same Aggregation, Other Syntax (~3 min)

Rewrite this summary — **average and max `years_homeless` per `education_level`** — using named aggregation (`.agg(name=(...))`) instead of the dict style.

In [ ]:
# For reference, dict style:
homeless.groupby("education_level").agg({"years_homeless": ["mean", "max"]})

# Your code here — same result, named-aggregation style


### Beyond the built-ins: custom aggregations

`.agg()` also accepts **any function**, including a `lambda` — useful when no built-in does exactly what you want. For example, the *range* (max − min) of support per city:

In [ ]:
homeless.groupby("city")["monthly_support_usd"].agg(lambda x: x.max() - x.min())

Groupby results carry the grouping column as an **index**. Use `.reset_index()` to turn it back into a normal column — useful before sorting, plotting, or merging with other data.

🎯 **Predict First:** which city do you think receives the highest **total** `monthly_support_usd`? Is that necessarily the same city with the highest **average**? Guess both, then check.

In [ ]:
# reset_index(), then sort to find the largest group
(
    homeless.groupby("city")["monthly_support_usd"]
    .sum()
    .reset_index()
    .sort_values("monthly_support_usd", ascending=False)
)

In [ ]:
# value_counts() is a shortcut for "groupby + count" on one column
homeless["education_level"].value_counts()

---
### 🔨 Task 2 — Group, Aggregate, Compare (~5 min)

1. Which `shelter_status` has the highest **average** `monthly_support_usd`?
2. Among people with more than 10 years homeless (`years_homeless > 10`), which `city` has the most people?
3. Using **named aggregation**, compute both the **count** of people and the **average** `years_homeless`, per `education_level`, in a single `.agg(...)` call.

*Hint for (2): filter first, then group.*

In [ ]:
# Your code here


---
## Careful with Aggregations

Aggregations are powerful, but every one of them **distorts** the data on purpose:

* **Counts** tell us *how many*, not *who*. 20 people in "shelter" in one city says nothing about their individual situations.
* **Means** smooth over differences. An average of 7 years homeless could mean everyone is close to 7 — or half the group is brand new and half has been homeless for over a decade. Two very different realities, one number.
* **Medians** resist outliers but hide them too — a median can look calm while a handful of extreme cases go completely unmentioned.
* **Sums** favor big groups. A large city might show the highest *total* support while a smaller city actually gives more *per person*.

Every aggregation **reduces detail** in exchange for a pattern you can see. That trade-off is not a flaw to fix — it's the whole point of grouping, and it's also exactly what **this Friday's Forum 1** is about: Chapter 1 of *Counting* by Deborah Stone argues that no summary number is "raw," because someone always had to decide what counts as alike before any counting could begin. `groupby()` is that decision, made in code.

---
## Coming Up

| Day | Topic | Builds on today |
|---|---|---|
| Wed | Choosing the right plot | The same cleaned dataset, now visualized — histograms, scatter, line, and bar charts |
| Fri | Forum 1 — *Counting*, Ch. 1 | What gets ignored when `groupby()` treats rows as "the same"? |
| Week 4 | Joining tables | Combining datasets *before* you can group or plot them together |